# AprovaEdu Analytics — Modelo Preditivo (diferencial)

Este notebook não faz parte das 4 perguntas obrigatórias — atende ao item de
diferencial "criação de score, segmentação ou modelo preditivo" do desafio.

**Pergunta**: dado o que sabemos sobre um aluno durante o ano de preparação
(presença, nota de diagnóstico, nota média em simulados, bolsa, perfil de
captação/origem), conseguimos estimar a probabilidade de aprovação no
vestibular?

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import modelo_preditivo as mp

df = mp.build_features()
df.head()


,aluno_id,ano,aulas_registradas,aulas_presente,taxa_presenca,aprovado,cidade,escola_origem,nota_diagnostico_media,bolsa_percentual_media,nota_simulados_media,n_simulados,canal_captacao
0,A00001,2023,76,65,0.8553,1,Fortaleza,Não informado,58.466667,13.333333,63.027778,18,Instagram
1,A00001,2024,58,52,0.8966,1,Fortaleza,Não informado,56.112500,20.000000,57.721429,14,Instagram
2,A00002,2025,67,61,0.9104,0,Crato,Federal,57.612500,5.625000,56.323529,17,Feira escolar
3,A00003,2022,88,78,0.8864,0,Horizonte,Privada,59.318182,10.000000,59.015789,19,Indicação
4,A00004,2021,68,58,0.8529,0,Juazeiro do Norte,Pública,54.640000,20.500000,59.473077,26,Google


## Preparação da base de modelagem

Cada linha é um (aluno, ano). O alvo é `aprovado` (0/1, aprovação no
vestibular naquele ano — com a mesma limitação de correspondência ano a ano
já discutida no relatório final). As features usam apenas informação
disponível **durante** o ano de preparação (não usam `nota_final_vestibular`
nem qualquer coluna de `aprovacoes_vestibular`, para evitar vazamento de
dados).

In [2]:
NUMERIC = mp.NUMERIC_FEATURES
CATEG = mp.CATEGORICAL_FEATURES
TARGET = mp.TARGET
print("Features numericas:", NUMERIC)
print("Features categoricas:", CATEG)

modelo_df = df.dropna(subset=NUMERIC + CATEG).copy()
print(f"\n{len(modelo_df)} linhas (de {len(df)}) apos remover nulos nas features")
print(f"Taxa de aprovacao na base do modelo: {modelo_df[TARGET].mean():.1%}")


Features numericas: ['taxa_presenca', 'nota_diagnostico_media', 'bolsa_percentual_media', 'nota_simulados_media']
Features categoricas: ['escola_origem', 'canal_captacao', 'cidade']

1022 linhas (de 1022) apos remover nulos nas features
Taxa de aprovacao na base do modelo: 33.2%


## Treino (regressão logística) e avaliação em conjunto de teste (25%, nunca visto no treino)

In [3]:
X = modelo_df[NUMERIC + CATEG]
y = modelo_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEG),
])
modelo = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
modelo.fit(X_train, y_train)

proba_test = modelo.predict_proba(X_test)[:, 1]
pred_test = modelo.predict(X_test)

print("Baseline (sempre prever a classe majoritaria):", round(1 - y_test.mean(), 4))
print("Acuracia do modelo:", round(accuracy_score(y_test, pred_test), 4))
print("AUC-ROC do modelo:", round(roc_auc_score(y_test, proba_test), 4))


Baseline (sempre prever a classe majoritaria): 0.668
Acuracia do modelo: 0.4492
AUC-ROC do modelo: 0.4585


In [4]:
print(confusion_matrix(y_test, pred_test))
print()
print(classification_report(y_test, pred_test, digits=3))


[[75 96]
 [45 40]]

              precision    recall  f1-score   support

           0      0.625     0.439     0.515       171
           1      0.294     0.471     0.362        85

    accuracy                          0.449       256
   macro avg      0.460     0.455     0.439       256
weighted avg      0.515     0.449     0.465       256



## Validação cruzada (5-fold) — para checar se o resultado é estável e não um artefato do split

In [5]:
cv_scores = cross_val_score(modelo, X, y, cv=5, scoring="roc_auc")
print("AUC por fold:", np.round(cv_scores, 4))
print(f"Media: {cv_scores.mean():.4f}  Desvio padrao: {cv_scores.std():.4f}")


AUC por fold: [0.4609 0.4927 0.5068 0.5006 0.4735]
Media: 0.4869  Desvio padrao: 0.0172


## Leitura honesta do resultado

O modelo tem **AUC-ROC ≈ 0,49** (média em 5-fold, desvio-padrão baixo — ou
seja, o resultado é estável, não é ruído do split). Um AUC de 0,5 equivale a
previsão aleatória; portanto, **o modelo não consegue prever aprovação melhor
que o acaso** usando presença, nota de diagnóstico, nota média em simulados,
bolsa, escola de origem, canal de captação e cidade.

Isso não é uma falha do modelo — é, na verdade, o mesmo achado da Pergunta 2
(correlação presença × aprovação ≈ 0) generalizado: **nesta base, os sinais
operacionais disponíveis não explicam quem é aprovado**. Duas leituras
possíveis para a coordenação:

1. **Os dados operacionais capturados hoje não incluem os fatores que
   realmente determinam a aprovação** (ex.: desempenho no dia da prova,
   preparo anterior ao ingresso no cursinho, concorrência específica da
   vaga/curso escolhido, fatores pessoais). Recomenda-se, se o objetivo for
   um modelo preditivo funcional, capturar sinais adicionais — por exemplo,
   a **evolução da nota do aluno ao longo do tempo** (tendência, não só
   média) e a **nota de simulados específicos ENEM** próximos à data da
   prova, que tendem a ser mais preditivos que a média geral.
2. **Reportar esse resultado à coordenação como achado, não como
   entrega de "score pronto".** Entregar um modelo com falso senso de
   precisão (ex.: dizendo "os alunos com nota X têm Y% de chance de
   aprovação" sem essa ressalva) seria enganoso e poderia levar a decisões
   ruins — por isso a abordagem correta aqui é reportar a ausência de sinal
   de forma transparente, em vez de forçar uma métrica de acurácia mais
   bonita sem validade real.

## Coeficientes do modelo (apenas para referência — sem poder preditivo real, ver conclusão acima)

In [6]:
feat_names = NUMERIC + list(
    modelo.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(CATEG)
)
coefs = modelo.named_steps["clf"].coef_[0]
coef_df = pd.DataFrame({"feature": feat_names, "coeficiente": coefs}).sort_values("coeficiente", ascending=False)
coef_df


,feature,coeficiente
16,cidade_Crato,0.446584
21,cidade_Juazeiro do Norte,0.318830
8,canal_captacao_Feira escolar,0.242698
10,canal_captacao_Indicação,0.221043
23,cidade_Pacatuba,0.162439
6,escola_origem_Privada,0.161691
14,cidade_Aquiraz,0.079543
1,nota_diagnostico_media,0.044947
9,canal_captacao_Google,0.044234
5,escola_origem_Não informado,0.043682
